# GutWise Baseline Eval — Gemma 4 E4B-it (Base)

Run 50 held-out IBS questions against the **base (unfinetuned)** Gemma 4 E4B-it.

Purpose: Establish baseline performance to measure GutWise fine-tuning improvement.

**Output**: `baseline_e4b_results.json` saved to Drive at `MyDrive/GutWise/eval/v1/`. Pair with `gutwise_v1_results_run{N}.json` for the offline Haiku judge.

**GPU**: L4 or better (bf16 required).

## 1. Install

In [1]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.15" peft accelerate bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 53.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 186.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 160.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 65.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 152.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 34.8 MB/s eta 0:00:00


In [2]:
import torch
gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram:.1f} GB")
assert torch.cuda.is_bf16_supported(), f"bf16 not supported on {gpu_name}"

GPU: NVIDIA L4
VRAM: 22.0 GB


## 2. Load base model

In [3]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    max_seq_length=1024,
    load_in_4bit=True,
    full_finetuning=False,
)
FastModel.for_inference(model)
print("Model loaded: unsloth/gemma-4-E4B-it")
print(f"VRAM: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Model loaded: unsloth/gemma-4-E4B-it
VRAM: 10.1 GB


## 3. Load eval prompts

Upload `datasets/eval/heldout_questions.jsonl` to `MyDrive/GutWise/eval/heldout_questions.jsonl`, or drag it into the Colab file browser.

In [4]:
import json
import os

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

EVAL_PATHS = [
    "/content/drive/MyDrive/GutWise/eval/heldout_questions.jsonl",
    "/content/heldout_questions.jsonl",
    "heldout_questions.jsonl",
]
prompts_path = next((p for p in EVAL_PATHS if os.path.exists(p)), None)
assert prompts_path, f"heldout_questions.jsonl not found in {EVAL_PATHS}"

with open(prompts_path) as f:
    prompts = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(prompts)} prompts from {prompts_path}")

SYSTEM_PROMPT = (
    "You are GutWise, an IBS health education assistant. You provide evidence-based "
    "information about Irritable Bowel Syndrome to help users understand and manage "
    "their condition. You are not a doctor and cannot diagnose or prescribe. Always "
    "recommend consulting a healthcare provider for personal medical decisions."
)

Mounted at /content/drive
Loaded 50 prompts from /content/drive/MyDrive/GutWise/eval/heldout_questions.jsonl


## 4. Eval loop

In [5]:
import time

MODEL_TAG = "gemma-4-E4B-it-base"
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.7

results = []
for i, p in enumerate(prompts):
    print(f"[{i+1}/{len(prompts)}] {p['id']} ({p['category']})...", end=" ", flush=True)
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": [{"type": "text", "text": p["question"]}]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    attention_mask = (inputs != tokenizer.pad_token_id).long()
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
        )
    elapsed = time.time() - start
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    results.append({
        "model": MODEL_TAG,
        "prompt_id": p["id"],
        "category": p["category"],
        "red_flag": p.get("red_flag", False),
        "expected_behavior": p.get("expected_behavior", ""),
        "prompt": p["question"],
        "response": response,
        "time_seconds": round(elapsed, 2),
        "temperature": TEMPERATURE,
        "error": None,
    })
    print(f"OK ({elapsed:.1f}s, {len(response)} chars)")

print(f"\nDone! {len(results)} responses generated.")

[1/50] eval_001 (factual_qa)... OK (200.2s, 4088 chars)
[2/50] eval_002 (factual_qa)... OK (102.4s, 2759 chars)
[3/50] eval_003 (factual_qa)... OK (107.5s, 3101 chars)
[4/50] eval_004 (factual_qa)... OK (90.2s, 2500 chars)
[5/50] eval_005 (factual_qa)... OK (110.2s, 3206 chars)
[6/50] eval_006 (factual_qa)... OK (83.1s, 2102 chars)
[7/50] eval_007 (factual_qa)... OK (83.3s, 2507 chars)
[8/50] eval_008 (factual_qa)... OK (83.4s, 2487 chars)
[9/50] eval_009 (factual_qa)... OK (98.1s, 2937 chars)
[10/50] eval_010 (factual_qa)... OK (96.2s, 2712 chars)
[11/50] eval_011 (factual_qa)... OK (97.3s, 2818 chars)
[12/50] eval_012 (factual_qa)... OK (75.3s, 2238 chars)
[13/50] eval_013 (factual_qa)... OK (158.4s, 4515 chars)
[14/50] eval_014 (factual_qa)... OK (108.8s, 3086 chars)
[15/50] eval_015 (factual_qa)... OK (80.6s, 2141 chars)
[16/50] eval_016 (factual_qa)... OK (120.7s, 3396 chars)
[17/50] eval_017 (factual_qa)... OK (114.6s, 3236 chars)
[18/50] eval_018 (factual_qa)... OK (99.7s, 3034 

## 5. Save results

In [6]:
output_name = "baseline_e4b_results.json"

with open(output_name, "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"Saved ./{output_name}")

drive_dir = "/content/drive/MyDrive/GutWise/eval/v1"
try:
    os.makedirs(drive_dir, exist_ok=True)
    drive_path = f"{drive_dir}/{output_name}"
    with open(drive_path, "w") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Also saved to {drive_path}")
except Exception as e:
    print(f"(Drive save skipped: {e})")

print("\nNext: run the GutWise v1 eval notebook, then judge both with scripts/evaluate/medical_judge.py")

Saved ./baseline_e4b_results.json
Also saved to /content/drive/MyDrive/GutWise/eval/v1/baseline_e4b_results.json

Next: run the GutWise v1 eval notebook, then judge both with scripts/evaluate/medical_judge.py


## 6. Preview

In [7]:
for r in results[:3]:
    print("=" * 70)
    print(f"[{r['prompt_id']}] ({r['category']})  red_flag={r['red_flag']}")
    print(f"Q: {r['prompt']}\n")
    print(f"A: {r['response'][:500]}{'...' if len(r['response']) > 500 else ''}")
    print()

[eval_001] (factual_qa)  red_flag=False
Q: What exactly is irritable bowel syndrome and how is it different from inflammatory bowel disease?

A: Hello! I'm GutWise, and I'm here to provide you with evidence-based information about Irritable Bowel Syndrome (IBS).

It's a very common question, as IBS and Inflammatory Bowel Disease (IBD) can sometimes share similar symptoms, which can certainly cause confusion.

Here is a detailed breakdown of what IBS is and how it differs from IBD:

***

### 🌿 What is Irritable Bowel Syndrome (IBS)?

**IBS is a functional gastrointestinal (GI) disorder.** This means that while a person with IBS experience...

[eval_002] (factual_qa)  red_flag=False
Q: Can you walk me through the Rome IV criteria used to diagnose IBS?

A: As GutWise, I can certainly walk you through the Rome IV criteria for Irritable Bowel Syndrome (IBS). It's important to remember that **I am an AI assistant and not a medical professional. This information is for educational purposes on